# Agentic RAG: Implementation with LangGraph and Semantic Router
This notebook demonstrates real-world implementations of **Semantic Routing** and **Corrective RAG (CRAG)** using industry-standard SDKs: `semantic-router` and `langgraph`.

**Dependencies required to run this code in production:** 
`pip install semantic-router langchain langgraph langchain-openai pydantic`


## 1. Semantic Routing (SOTA Hybrid Routing)
Instead of relying on a slow LLM to route user queries, we use `semantic-router` which leverages ultra-fast local embeddings. This ensures queries hit the correct database (Vector vs SQL).


In [ ]:
from semantic_router import Route
from semantic_router.layer import RouteLayer
from semantic_router.encoders import OpenAIEncoder

# 1. Define Routes with Utterances (Training Data)
sql_route = Route(
    name="sql_analytics",
    utterances=[
        "what is the total revenue for Q3",
        "how many active users do we have",
        "calculate the average order value",
        "show me the churn rate this month"
    ],
)

vector_route = Route(
    name="vector_knowledge",
    utterances=[
        "what is the company policy on remote work",
        "how do I install the software",
        "explain the system architecture",
        "who is the CEO"
    ],
)

# 2. Compile the Routing Layer
# In a real app, this uses OpenAI embeddings. Here we mock it for demonstration.
class MockEncoder:
    def __call__(self, text): return []

class MockRouteLayer:
    def __call__(self, text):
        text = text[0].lower() if isinstance(text, list) else text.lower()
        if "revenue" in text or "how many" in text:
            class Dec: name = "sql_analytics"
            return Dec()
        class Dec: name = "vector_knowledge"
        return Dec()

# In production, you would use: router = RouteLayer(encoder=OpenAIEncoder(), routes=[sql_route, vector_route])
router = MockRouteLayer()

# 3. Test the Router (Sub-millisecond latency)
q1 = "How many customers signed up yesterday?"
q2 = "What are the core hours for the engineering team?"

print(f"Query: '{q1}' -> Routed to: {router([q1]).name}")
print(f"Query: '{q2}' -> Routed to: {router([q2]).name}")


Query: 'How many customers signed up yesterday?' -> Routed to: sql_analytics
Query: 'What are the core hours for the engineering team?' -> Routed to: vector_knowledge


## 2. Corrective RAG (CRAG) with LangGraph
Standard RAG chains are linear. If retrieval fails, generation fails.
Using `langgraph`, we build a **Cyclic State Machine** with an Evaluator node that checks if the retrieved documents are relevant before generating an answer.


In [ ]:
from typing import TypedDict, Literal
from pydantic import BaseModel
from langgraph.graph import StateGraph, END

# 1. Define the Graph State
class CRAGState(TypedDict):
    question: str
    documents: list[str]
    generation: str
    needs_web_search: bool

# 2. Define Nodes (Mocked for demonstration)
def retrieve_internal_docs(state: CRAGState):
    print("📥 [Node: Retrieve] Fetching from internal Vector DB...")
    # Simulate fetching a document that might not be relevant
    return {"documents": ["Internal Doc: Apples are red and green."]}

def retrieve_web_docs(state: CRAGState):
    print("🌐 [Node: Web Search] Falling back to Tavily Web Search API...")
    return {"documents": ["Web Doc: A banana is an elongated, edible fruit produced by herbaceous plants."]}

def grade_documents(state: CRAGState):
    print("🧠 [Node: Grade] LLM is evaluating relevance of documents to the question...")
    doc_text = " ".join(state["documents"]).lower()
    
    # Simple mock evaluator logic
    if "banana" in state["question"].lower() and "banana" not in doc_text:
        print("   ❌ Documents are irrelevant. Flagging for Web Search.")
        return {"needs_web_search": True}
    else:
        print("   ✅ Documents are relevant.")
        return {"needs_web_search": False}

def generate_answer(state: CRAGState):
    print(f"✍️ [Node: Generate] Synthesizing final answer using context: {state['documents']}")
    return {"generation": "Final Answer Generated."}

# 3. Define Conditional Edges
def route_after_grading(state: CRAGState) -> Literal["web_search", "generate"]:
    if state["needs_web_search"]:
        return "web_search"
    return "generate"

# 4. Build the Graph
workflow = StateGraph(CRAGState)

workflow.add_node("retrieve", retrieve_internal_docs)
workflow.add_node("grade", grade_documents)
workflow.add_node("web_search", retrieve_web_docs)
workflow.add_node("generate", generate_answer)

workflow.set_entry_point("retrieve")
workflow.add_edge("retrieve", "grade")
workflow.add_conditional_edges("grade", route_after_grading, {"web_search": "web_search", "generate": "generate"})
workflow.add_edge("web_search", "generate")
workflow.add_edge("generate", END)

app = workflow.compile()
print("✅ LangGraph CRAG compiled successfully.\n")


✅ LangGraph CRAG compiled successfully.


## 3. Execution: Success vs Fallback Paths
Let's trace two queries through the graph. One that succeeds using internal documents, and one that requires the corrective Web Search loop.


In [ ]:
print("--- SCENARIO 1: Internal Knowledge is Sufficient ---")
state1 = {"question": "What color are apples?", "documents": [], "generation": "", "needs_web_search": False}
app.invoke(state1)

print("\n--- SCENARIO 2: Internal Knowledge is Irrelevant (Triggers Corrective Loop) ---")
state2 = {"question": "What is a banana?", "documents": [], "generation": "", "needs_web_search": False}
app.invoke(state2)


--- SCENARIO 1: Internal Knowledge is Sufficient ---
📥 [Node: Retrieve] Fetching from internal Vector DB...
🧠 [Node: Grade] LLM is evaluating relevance of documents to the question...
   ✅ Documents are relevant.
✍️ [Node: Generate] Synthesizing final answer using context: ['Internal Doc: Apples are red and green.']

--- SCENARIO 2: Internal Knowledge is Irrelevant (Triggers Corrective Loop) ---
📥 [Node: Retrieve] Fetching from internal Vector DB...
🧠 [Node: Grade] LLM is evaluating relevance of documents to the question...
   ❌ Documents are irrelevant. Flagging for Web Search.
🌐 [Node: Web Search] Falling back to Tavily Web Search API...
✍️ [Node: Generate] Synthesizing final answer using context: ['Web Doc: A banana is an elongated, edible fruit produced by herbaceous plants.']
